# AXIOM Ephemeral GPU Inference Lab — Colab

This notebook is a **disposable learning and evidence environment**, not an AXIOM node and not an authorized model-provider adapter.

It loads an open model on whatever CUDA accelerator Colab actually assigns and emits a small inert observation. Hardware types and usage limits are dynamic rather than assumed.

Reference: https://research.google.com/colaboratory/faq.html

**Do not put secrets, private AXIOM state, personal data, production credentials, or mounted Drive data into this notebook.**


In [ ]:
import hashlib
import json
import platform
import subprocess
import sys
import time
import uuid
from datetime import datetime, timezone

import torch

def utc_now():
    return datetime.now(timezone.utc).isoformat().replace('+00:00', 'Z')

def sha256_text(value):
    return hashlib.sha256(value.encode('utf-8')).hexdigest()

if not torch.cuda.is_available():
    raise RuntimeError('No CUDA GPU is assigned. Colab GPU availability is not guaranteed; stop rather than silently falling back.')

props = torch.cuda.get_device_properties(0)
hardware = {
    'gpu_name': torch.cuda.get_device_name(0),
    'memory_total_gb': round(props.total_memory / (1024 ** 3), 2),
    'compute_capability': f'{props.major}.{props.minor}',
    'python': sys.version.split()[0],
    'platform': platform.platform(),
    'torch': torch.__version__,
    'cuda': torch.version.cuda,
}
print(json.dumps(hardware, indent=2, sort_keys=True))
print('\nnvidia-smi:')
subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,driver_version', '--format=csv,noheader'], check=False)


## Notebook-only inference dependencies

These packages exist only in the disposable Colab VM. This does not change the AXIOM kernel dependency set.


In [ ]:
%pip -q install "transformers>=4.45" "accelerate>=1.0" sentencepiece


In [ ]:
import transformers
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_ID = 'Qwen/Qwen2.5-0.5B-Instruct'

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=False)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype='auto',
    device_map='auto',
    trust_remote_code=False,
)
model.eval()
model_revision = getattr(model.config, '_commit_hash', None)
print({'model_ref': MODEL_ID, 'model_revision': model_revision, 'transformers': transformers.__version__})


## One bounded synthetic request

The request below is deliberately synthetic. The notebook accepts no AXIOM credentials and exposes no HTTP, SSH, tunnel, web UI, or distributed worker.


In [ ]:
request = {
    "schema": "axiom-ephemeral-inference-request.v0",
    "request_id": f"lab-{uuid.uuid4()}",
    "purpose": "synthetic-lab-inference",
    "prompt": "Return exactly one sentence explaining that an ephemeral GPU can be useful for learning local model integration.",
    "max_new_tokens": 96,
    "authority_effect": "none",
}

allowed_fields = {"schema", "request_id", "purpose", "prompt", "max_new_tokens", "authority_effect"}
if set(request) != allowed_fields:
    raise ValueError('request contains unsupported fields')
if request['schema'] != 'axiom-ephemeral-inference-request.v0':
    raise ValueError('request schema mismatch')
if request['purpose'] != 'synthetic-lab-inference' or request['authority_effect'] != 'none':
    raise ValueError('request boundary violation')
if not isinstance(request['max_new_tokens'], int) or not 1 <= request['max_new_tokens'] <= 128:
    raise ValueError('max_new_tokens must be in 1..128')
if not isinstance(request['prompt'], str) or not 1 <= len(request['prompt']) <= 1000:
    raise ValueError('prompt must contain 1..1000 characters')

messages = [{'role': 'user', 'content': request['prompt']}]
rendered = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer([rendered], return_tensors='pt').to(model.device)
torch.cuda.reset_peak_memory_stats()
started_at = utc_now()
start = time.perf_counter()
with torch.inference_mode():
    generated = model.generate(**inputs, max_new_tokens=request['max_new_tokens'], do_sample=False)
elapsed_ms = round((time.perf_counter() - start) * 1000, 2)
completed_at = utc_now()
new_tokens = generated[0][inputs.input_ids.shape[1]:]
output_text = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
print(output_text)


In [ ]:
observation = {
    "schema": "axiom-ephemeral-inference-observation.v0",
    "experiment_id": "lab-010-ephemeral-gpu-inference",
    "request_id": request["request_id"],
    "purpose": request["purpose"],
    "model_ref": MODEL_ID,
    "model_revision": model_revision,
    "input_sha256": sha256_text(request["prompt"]),
    "output_sha256": sha256_text(output_text),
    "environment": {**hardware, "transformers": transformers.__version__, "runtime_class": "google-colab-managed-runtime"},
    "metrics": {
        "elapsed_ms": elapsed_ms,
        "input_tokens": int(inputs.input_ids.shape[1]),
        "output_tokens": int(new_tokens.shape[0]),
        "peak_gpu_memory_gb": round(torch.cuda.max_memory_allocated() / (1024 ** 3), 3),
    },
    "started_at": started_at,
    "completed_at": completed_at,
    "boundaries": {
        "authority_effect": "none",
        "credential_visibility": "none",
        "durable_state_effect": "ephemeral-only",
        "network_listener": False,
        "provider_registration": False,
        "gateway_route_created": False,
        "production_claim": False,
    },
    "notes": [
        "Hardware and free-tier quotas are observed at runtime and are not assumed.",
        "The observation contains hashes and measurements, not the prompt or model output.",
        "This laboratory does not register Colab as an AXIOM provider or compute node.",
    ],
}
OUTPUT_PATH = '/content/axiom-ephemeral-inference-observation.json'
with open(OUTPUT_PATH, 'w', encoding='utf-8') as handle:
    json.dump(observation, handle, indent=2, sort_keys=True)
    handle.write('\n')
print(json.dumps(observation, indent=2, sort_keys=True))
print(f'\nSaved ephemeral observation to {OUTPUT_PATH}')


## Bring the result back into AXIOM development

The JSON observation is inert. It may be used as a development fixture, but it is **not** a capability grant, provider registration, runtime receipt, benchmark certification, or remote-execution proof.

Disconnect and delete the Colab runtime when finished. Treat the runtime as disposable.
